### Chapter 10: Tool Engineering
### Topic 5: Error, Timeout, and Retry Handling

# End-to-End Flow of Error Handling Fundamentals

This chapter explains how a production tool should handle different outcomes.

The most important idea is that **not every unsuccessful lookup is an error**. A tool must clearly distinguish between a valid business outcome and a genuine system failure.

The program performs the following steps in order:

1. The LLM requests a tool.
2. The application executes the tool.
3. The tool performs the lookup.
4. The tool determines the outcome.
5. The tool returns a structured response.
6. The application sends the result back to the LLM.
7. The LLM decides what to do next.

This chapter focuses on **understanding different types of outcomes**, not retrying failed requests.

---

### Why Do We Need Error Handling?

Suppose the customer asks:

> **"Check my FD reference FD123456."**

The LLM requests:

```text
validate_fd_reference(FD123456)
```

The tool performs the lookup.

Now several different outcomes are possible.

A production tool must tell the LLM exactly **what happened**.

Otherwise, the LLM cannot generate the correct response.

---

### What Are the Possible Outcomes?

A lookup can end in four different ways.

#### Outcome 1: Record Found

The lookup succeeds.

Example:

```text
Reference

FD123456

Status

Active
```

The tool returns:

```python
{
    "found": True,
    "status": "Active"
}
```

The LLM now has the required information.

---

#### Outcome 2: Record Not Found

The lookup succeeds.

But no matching record exists.

Example:

```text
Reference

FD999999
```

The tool returns:

```python
{
    "found": False,
    "reference_number": "FD999999"
}
```

Notice something important.

The lookup worked correctly.

The answer simply happens to be:

> **No matching record exists.**

This is **not an error**.

---

#### Outcome 3: Temporary System Failure

Suppose the database is temporarily unavailable.

Example:

```text
Database Timeout
```

The tool cannot determine whether the reference exists.

It returns something like:

```python
{
    "error": "lookup_unavailable",
    "reference_number": "FD123456"
}
```

Now the LLM understands:

> The lookup could not be completed.

This is different from:

```text
found = False
```

---

#### Outcome 4: Permanent Failure

Suppose the request itself is invalid.

Example:

```text
reference_number = ""
```

or

```text
reference_number = ABC
```

The tool cannot perform a meaningful lookup.

Retrying with the same input will never succeed.

This is a permanent failure.

---

### Why Is "Not Found" Not an Error?

This is one of the most important concepts.

Suppose the customer enters:

```text
FD999999
```

The database searches every record.

Nothing matches.

The database successfully answered the question.

The answer is simply:

```text
No Record Found
```

The system worked perfectly.

Returning:

```text
found = False
```

is the correct behavior.

Treating this as an error would be incorrect.

---

### Why Is a Temporary Failure Different?

Now suppose the database crashes.

Example:

```text
Connection Timeout
```

Can the tool say:

```text
found = False
```

No.

The tool never completed the lookup.

It has no idea whether the record exists.

Instead, it must honestly report:

```text
Lookup Unavailable
```

This allows the LLM to respond appropriately.

---

### Why Is a Permanent Failure Different?

Suppose the input is:

```text
reference_number = ""
```

Should the tool retry?

No.

The problem is not the database.

The problem is the request itself.

Repeating exactly the same request will always fail.

The correct response is to return an input validation error.

---

### Comparing the Four Outcomes

Suppose the customer enters a reference number.

The tool performs the lookup.

Possible results:

```text
Record Exists

↓

found = True
```

---

```text
Record Does Not Exist

↓

found = False
```

---

```text
Database Unavailable

↓

error = lookup_unavailable
```

---

```text
Invalid Request

↓

validation_error
```

Each outcome represents a different situation.

They should never be combined into a single result like:

```text
Failure
```

---

### Why Is Structured Error Information Important?

Suppose every failure returned:

```text
False
```

What happened?

Did the customer type the wrong reference?

Was the database unavailable?

Was the input invalid?

Nobody knows.

Instead, the tool should clearly identify the outcome.

Example:

```python
{
    "error": "lookup_unavailable"
}
```

or

```python
{
    "validation_error": "invalid_reference_number"
}
```

Now the application and the LLM know exactly what happened.

---

### What Does the LLM Do With Different Results?

Suppose the tool returns:

```python
{
    "found": False
}
```

The LLM replies:

> "The reference number was not found."

---

Suppose the tool returns:

```python
{
    "error": "lookup_unavailable"
}
```

The LLM replies:

> "I'm currently unable to verify your FD because the system is temporarily unavailable. Please try again later."

These are completely different customer responses.

---

### How Is This Implemented in Our Project?

Our current implementation already handles:

```python
found = True
```

and

```python
found = False
```

correctly.

The next improvement is to introduce a third category.

Example:

```python
{
    "error": "lookup_unavailable"
}
```

Now the LLM can distinguish:

- Record exists.
- Record does not exist.
- Lookup could not be completed.

This makes the agent much more reliable.

---

### Why Doesn't the Agent Crash?

Suppose the database becomes unavailable.

Instead of crashing,

the tool returns a structured error.

The application sends this result back to the LLM just like any other tool result.

The LLM can continue reasoning.

It may:

- Inform the customer.
- Suggest trying again later.
- Escalate to a human agent.

The conversation continues instead of terminating unexpectedly.

---

### Production Engineer's Perspective

A production tool should distinguish between four outcomes:

| Situation | Tool Response |
|-----------|---------------|
| Record exists | `found = True` |
| Record does not exist | `found = False` |
| Temporary system failure | `error = lookup_unavailable` |
| Invalid request | `validation_error` |

The key principle is:

- **Business outcomes are not errors.**
- **System failures are errors.**
- **Invalid inputs are validation failures.**

Keeping these categories separate allows the LLM, the application, and production monitoring systems to respond correctly without confusing one situation with another.

---

### Lead-Level Interview Questions

**Basic**

- Q: Why shouldn't a tool's legitimate `found: False` result be treated as an error requiring retry?  
  A: `found: False` means the underlying system worked correctly and gave a definitive, valid answer — the reference number genuinely doesn't exist. Retrying wouldn't change this outcome, since nothing about the system's state was actually uncertain or incomplete; it was a successful lookup that happened to return a negative result, not a failure to complete the lookup at all.

- Q: What's the difference between a transient error and a permanent (non-retryable) error, and why does this distinction matter for retry logic?  
  A: A transient error (like a brief network timeout) has a real chance of succeeding if attempted again, since the underlying problem may resolve on its own. A permanent error (like a fundamentally malformed request) will fail identically no matter how many times it's retried. Retrying a permanent error wastes time and calls with zero chance of a different outcome — this distinction is exactly why malformed-input validation (Topic 1) should happen before any retry-eligible code path, not be conflated with it.

**Intermediate**

- Q: What should happen when a tool's retry attempts are all exhausted without success?  
  A: The tool should return a distinct, honest, explicitly-labeled error result — different in shape from both a `found: True` success and a `found: False` legitimate negative — communicating that the check could not be completed at all. This result gets fed back to the model exactly like any other tool result, so the model can reason accordingly (for example, informing the customer of a temporary system issue) rather than either crashing the whole request or fabricating a plausible-sounding but unfounded answer.

- Q: Why does waiting between retry attempts matter, rather than retrying immediately and repeatedly?  
  A: Retrying immediately and repeatedly against a dependency that's failing because it's overloaded can make the underlying problem worse by adding more load to an already-struggling system. Waiting between attempts — especially with an increasing delay — gives a genuinely transient problem real room to resolve before the next attempt, improving the odds that a retry actually succeeds rather than just repeating the same failure faster.

**Advanced**

- Q: Design the full error-handling flow for a tool call, from the moment the underlying dependency fails to the moment the model sees a result, distinguishing all the relevant failure categories.  
  A: First, dispatch-layer validation (Topic 1) checks the request's basic well-formedness before any real execution — a malformed request is rejected here, never reaching retry logic at all. If the request passes validation and the real function executes, a genuine transient error (like a timeout) triggers the retry mechanism: wait, attempt again, up to a maximum count, with increasing delay between attempts. If a retry succeeds, return that result normally. If all retries are exhausted, construct a distinct, explicitly-labeled error result (not a fabricated `found: False`, not a crash) and feed it back to the model as a `tool_result`, using the same message mechanics as any successful call (Topic 2) — the model must see this failure state explicitly in its own context to reason about it correctly.

- Q: A tool experiences a rising rate of exhausted-retry errors over time. How would you decide whether to adjust the retry policy (more attempts, longer waits) versus investigating the underlying dependency itself?  
  A: Adjusting the retry policy treats the symptom, and is only appropriate if the underlying failures are genuinely brief and self-resolving, just needing slightly more patience to catch on a retry. A rising rate over time, rather than a stable, low background rate, is a signal the underlying dependency itself has a real, worsening reliability problem — more retries would only mask this, delaying detection while making the customer wait even longer for calls that are increasingly likely to fail regardless. The retry logging discussed in this topic (attempt counts, timing, ultimate outcomes) is exactly the evidence needed to distinguish which situation is actually occurring.

**Scenario-based**

- Q: In production, customers occasionally report unusually long waits for an FD status check, and some report the assistant said it "couldn't check the account status right now." Walk through your investigation.  
  A: The explicit "couldn't check right now" message is the intended behavior for the exhausted-retries error case, not a bug in itself — the actual investigation should focus on how *often* this is happening and why. Check the retry logs for this tool specifically: is the underlying dependency failing more often than expected, and is the current retry count/delay policy appropriate given how long transient failures typically take to resolve, or is it retrying too few times too quickly, or too many times with too much cumulative delay before giving up? This log-driven evidence should directly inform whether the fix is tuning the retry policy or escalating an investigation into the underlying dependency's reliability itself.

**Follow-up questions to expect:**

- "How would you decide the right maximum retry count and delay for a specific tool's dependency?"
- "What would you log to distinguish a transient dependency issue from a persistent one?"


# End-to-End Flow of Production Retry Engineering

This chapter explains how production systems recover from temporary tool failures.

The previous chapter explained the different outcomes a tool can return. This chapter focuses on **what to do when a tool fails because of a temporary system problem**.

The program performs the following steps in order:

1. Execute the tool.
2. Check whether it succeeded.
3. If a temporary error occurs, wait for a short time.
4. Retry the tool.
5. Stop retrying after the maximum number of attempts.
6. Return either the successful result or an explicit error.
7. Log and monitor every retry attempt.

The goal is to improve reliability **without increasing latency unnecessarily**.

---

### Why Do We Need Retry Logic?

Suppose the customer asks:

> **"Check my FD reference FD123456."**

The tool tries to query the database.

Instead of returning a result, the database responds:

```text
Connection Timeout
```

Should the application immediately fail?

Not necessarily.

Sometimes the database is temporarily busy.

Trying again after a short delay may succeed.

This is why production systems implement **retry logic**.

---

### When Should We Retry?

Retry only when the failure is **temporary**.

Examples:

```text
Database Timeout

Network Timeout

Temporary Database Unavailable

Temporary Network Failure
```

These problems may disappear after a few seconds.

Retrying has a reasonable chance of succeeding.

---

### When Should We NOT Retry?

Suppose the input is:

```text
reference_number = ""
```

or

```text
reference_number = ABC123
```

Retrying the same request gives exactly the same result.

The input is invalid.

Retrying only wastes time.

Another example:

```text
Record Not Found
```

This is not an error.

The lookup completed successfully.

Running the same lookup again will still return:

```text
Not Found
```

Retrying provides no benefit.

---

### Complete Retry Flow

Suppose the tool executes.

Four outcomes are possible.

```text
Tool Executes

↓

Record Found

↓

Return Result
```

---

```text
Tool Executes

↓

Record Not Found

↓

Return Result
```

---

```text
Tool Executes

↓

Temporary Failure

↓

Retry
```

---

```text
Tool Executes

↓

Permanent Failure

↓

Return Error
```

Notice that only one outcome enters the retry path.

---

### How Does Retry Work?

Suppose the database times out.

The application performs:

```text
Attempt 1

↓

Timeout

↓

Wait

↓

Attempt 2

↓

Timeout

↓

Wait

↓

Attempt 3

↓

Success
```

The successful result is immediately returned to the LLM.

The customer never knows that retries occurred.

---

### Why Wait Between Retries?

Suppose the database is overloaded.

Immediately retrying:

```text
Attempt

↓

Timeout

↓

Immediate Retry

↓

Timeout

↓

Immediate Retry

↓

Timeout
```

adds even more pressure to an already struggling database.

Instead, wait before retrying.

```text
Attempt

↓

Timeout

↓

Wait

↓

Retry
```

The waiting period gives the dependency time to recover.

---

### What Is Increasing Delay?

Instead of waiting the same amount every time,

the application can gradually increase the delay.

Example:

```text
Attempt 1

↓

Wait

1 Second

------------------------

Attempt 2

↓

Wait

2 Seconds

------------------------

Attempt 3

↓

Wait

4 Seconds
```

This approach is called **Increasing Delay** (commonly known as exponential backoff).

It reduces pressure on overloaded systems.

---

### Why Do We Need a Maximum Retry Limit?

Suppose the database remains unavailable.

Without a limit:

```text
Retry

↓

Retry

↓

Retry

↓

Retry

↓

Forever
```

The customer never receives a response.

Production systems always define a limit.

Example:

```text
Maximum Attempts

3
```

After three unsuccessful attempts, the application stops retrying.

---

### What Happens After All Retries Fail?

Suppose every attempt fails.

The application should not return:

```text
found = False
```

because the lookup never completed.

Instead, return an explicit error.

Example:

```python
{
    "error": "lookup_unavailable",
    "reference_number": "FD123456"
}
```

The application sends this result to the LLM.

The LLM can now respond honestly.

Example:

> "I'm currently unable to verify your FD because the verification system is temporarily unavailable. Please try again later."

---

### Why Is This Better Than Crashing?

Suppose the application crashes.

The customer receives:

```text
Internal Server Error
```

The conversation is lost.

Instead, the tool returns a structured error.

The LLM continues the conversation and explains the situation clearly.

The customer receives a much better experience.

---

### Why Should Every Retry Be Logged?

Suppose the third attempt succeeds.

Without logging, engineers only see:

```text
Success
```

They never know that two earlier attempts failed.

Instead, log every attempt.

Example:

```text
Attempt

1

Status

Timeout

------------------------

Attempt

2

Status

Timeout

------------------------

Attempt

3

Status

Success
```

This helps identify unstable systems before they become major incidents.

---

### What Should We Monitor?

Production systems monitor:

- Number of retries
- Retry success rate
- Retry failure rate
- Average retry attempts
- Average retry latency
- Maximum retry latency
- Timeout frequency

Example:

```text
validate_fd_reference

Calls

20,000

Retry Rate

2%

Average Retry Attempts

1.3

Average Latency

18 ms
```

These metrics help identify:

- Unstable databases
- Network problems
- Slow dependencies

---

### Why Does Retry Increase Latency?

Suppose a lookup normally takes:

```text
20 ms
```

Now suppose three retries occur.

```text
Attempt 1

20 ms

↓

Wait

100 ms

↓

Attempt 2

20 ms

↓

Wait

200 ms

↓

Attempt 3

20 ms
```

The total response time becomes much larger.

Retry improves reliability,

but increases worst-case latency.

This trade-off must be carefully balanced.

---

### Why Does Retry Increase Cost?

Each retry:

- Executes the tool again.
- Uses additional infrastructure.
- Increases processing time.

For external APIs,

each retry may also increase billing costs.

Production systems should retry only when the probability of recovery justifies the additional cost.

---

### Where Should Retry Logic Be Implemented?

There are two common approaches.

#### Inside Every Tool

Each tool implements its own retry policy.

Advantages:

- Tool-specific behavior.
- Greater flexibility.

---

#### Shared Retry Wrapper

The application automatically retries every tool.

Advantages:

- Less duplicate code.
- Consistent behavior.

---

### Production Recommendation

Use a shared retry mechanism for common behavior.

Allow individual tools to override it when they have special requirements.

---

### Common Production Mistakes

#### Mistake 1

Retrying:

```text
Record Not Found
```

This is not an error.

---

#### Mistake 2

Retrying invalid inputs.

The request will fail every time.

---

#### Mistake 3

Retrying forever.

Always define a maximum retry limit.

---

#### Mistake 4

Retrying immediately without waiting.

This can overload already struggling systems.

---

#### Mistake 5

Returning:

```text
found = False
```

after retries fail.

The lookup never completed.

Return an explicit error instead.

---

#### Mistake 6

Not logging retry attempts.

Successful retries may hide underlying reliability problems.

---

### Production Engineer's Perspective

Retry logic should recover from **temporary failures**, not hide permanent ones.

A production retry mechanism should:

- Retry only transient failures.
- Never retry successful lookups.
- Never retry "Not Found" results.
- Never retry invalid requests.
- Wait between attempts.
- Limit the maximum number of retries.
- Return an explicit error if recovery fails.
- Log every retry attempt.
- Monitor retry metrics continuously.

Think of retry as a recovery strategy—not as a solution for every failure.

The purpose of retry is simple:

> **Give temporary system problems a chance to recover while keeping the customer experience reliable, honest, and predictable.**

---

### Module 1: Three Failure Categories, Modeled Distinctly

Build a simulated dependency that can produce all three failure categories (legitimate negative, transient error, permanent error), and show why each needs different handling.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Three failure categories, modeled distinctly
# ------------------------------------------------------------------

import time

class TransientError(Exception):
    """A real, distinguishable exception type for transient failures --
    e.g. a timeout or brief network issue. Retrying THIS category has
    a real chance of succeeding."""
    pass

class PermanentError(Exception):
    """A real, distinguishable exception type for non-retryable
    failures -- e.g. malformed input. Retrying THIS category is
    pointless; it will fail identically every time."""
    pass


FD_RECORDS_TABLE = {
    "BJ2019FD7717": {"Customer_Name": "Shobha Chopra", "Status": "Closed (Premature)"},
}

# a controllable simulated dependency: fails a specified number of
# times with a TransientError, then succeeds -- lets us test retry
# logic against a REAL, controllable failure pattern
class SimulatedFlakyDependency:
    def __init__(self, fail_count: int, permanent_failure: bool = False):
        self.fail_count = fail_count
        self.permanent_failure = permanent_failure
        self.attempts_made = 0

    def query(self, reference_number: str) -> dict:
        self.attempts_made += 1
        if self.permanent_failure:
            raise PermanentError(f"Malformed query for '{reference_number}' -- will fail every time.")
        if self.attempts_made <= self.fail_count:
            raise TransientError(f"Attempt {self.attempts_made}: connection timed out.")
        # success -- either a real record or a legitimate not-found
        record = FD_RECORDS_TABLE.get(reference_number)
        if record is None:
            return {"found": False, "reference_number": reference_number}
        return {"found": True, "reference_number": reference_number, **record}


print("=" * 70)
print("MODULE 1: THREE FAILURE CATEGORIES")
print("=" * 70)

# Category 1: legitimate negative result -- NOT an error at all
reliable_dep = SimulatedFlakyDependency(fail_count=0)
legit_negative = reliable_dep.query("BJ9999FD0000")
print(f"\n[Category 1: legitimate negative result]")
print(f"  Result: {legit_negative}")
print(f"  This is NOT an error -- no retry needed, the answer is definitively 'no'.")

# Category 2: transient error -- demonstrated by raising, then catching
transient_dep = SimulatedFlakyDependency(fail_count=2)
print(f"\n[Category 2: transient error -- will fail twice, then succeed]")
try:
    transient_dep.query("BJ2019FD7717")
except TransientError as e:
    print(f"  Attempt 1 raised: {e}")

# Category 3: permanent error -- will NEVER succeed no matter how many attempts
permanent_dep = SimulatedFlakyDependency(fail_count=0, permanent_failure=True)
print(f"\n[Category 3: permanent error -- will fail EVERY attempt, forever]")
try:
    permanent_dep.query("malformed-input")
except PermanentError as e:
    print(f"  Attempt raised: {e}")
    print(f"  Retrying this would be POINTLESS -- it will fail identically every time.")

print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: THREE FAILURE CATEGORIES

[Category 1: legitimate negative result]
  Result: {'found': False, 'reference_number': 'BJ9999FD0000'}
  This is NOT an error -- no retry needed, the answer is definitively 'no'.

[Category 2: transient error -- will fail twice, then succeed]
  Attempt 1 raised: Attempt 1: connection timed out.

[Category 3: permanent error -- will fail EVERY attempt, forever]
  Attempt raised: Malformed query for 'malformed-input' -- will fail every time.
  Retrying this would be POINTLESS -- it will fail identically every time.

Module 1 complete. Run Module 2 next.


### Module 2: Real Retry-With-Backoff Logic, Measured Against the Simulated Transient Failure

Implement actual retry logic with increasing delay, and measure the REAL time it takes to succeed against a dependency that fails a controlled number of times first.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Real retry-with-backoff, measured against real timing
# ------------------------------------------------------------------

def call_with_retry(query_func, reference_number: str, max_attempts: int = 4,
                     base_delay_seconds: float = 0.2) -> dict:
    """REAL retry-with-increasing-delay logic. Only retries on
    TransientError -- a PermanentError is NOT retried at all, since
    retrying it can never succeed (Category 3 from Module 1)."""
    last_exception = None
    for attempt in range(1, max_attempts + 1):
        try:
            result = query_func(reference_number)
            return {"status": "success", "result": result, "attempts_used": attempt}
        except TransientError as e:
            last_exception = e
            if attempt < max_attempts:
                delay = base_delay_seconds * attempt  # INCREASING delay each attempt
                time.sleep(delay)
        except PermanentError as e:
            # NOT retried -- fail immediately, exactly per Category 3's principle
            return {"status": "permanent_failure", "error": str(e), "attempts_used": attempt}

    return {"status": "retries_exhausted", "error": str(last_exception), "attempts_used": max_attempts}


print("=" * 70)
print("REAL RETRY-WITH-BACKOFF, MEASURED TIMING")
print("=" * 70)

# Case A: transient failure that RESOLVES within the retry budget
dep_resolves = SimulatedFlakyDependency(fail_count=2)
start = time.perf_counter()
result_a = call_with_retry(dep_resolves.query, "BJ2019FD7717", max_attempts=4)
elapsed_a = time.perf_counter() - start

print(f"\n[Case A: fails twice, succeeds on 3rd attempt]")
result_a_status = result_a["status"]
print(f"  Status: {result_a_status}")
result_a_attempts_used = result_a["attempts_used"]
print(f"  Attempts used: {result_a_attempts_used}")
print(f"  Real elapsed time: {elapsed_a:.3f}s (includes REAL backoff delays)")
if result_a["status"] == "success":
    result_a_result = result_a["result"]
    print(f"  Result: {result_a_result}")

# Case B: transient failure that NEVER resolves within the retry budget
dep_never_resolves = SimulatedFlakyDependency(fail_count=10)  # fails more than max_attempts
start = time.perf_counter()
result_b = call_with_retry(dep_never_resolves.query, "BJ2019FD7717", max_attempts=4)
elapsed_b = time.perf_counter() - start

print(f"\n[Case B: fails 10 times -- exceeds retry budget of 4 attempts]")
result_b_status = result_b["status"]
print(f"  Status: {result_b_status}")
result_b_attempts_used = result_b["attempts_used"]
print(f"  Attempts used: {result_b_attempts_used}")
print(f"  Real elapsed time: {elapsed_b:.3f}s (full retry budget consumed)")

# Case C: PERMANENT error -- should NOT retry at all, should fail fast
dep_permanent = SimulatedFlakyDependency(fail_count=0, permanent_failure=True)
start = time.perf_counter()
result_c = call_with_retry(dep_permanent.query, "malformed-input", max_attempts=4)
elapsed_c = time.perf_counter() - start

print(f"\n[Case C: PERMANENT error -- should fail FAST, no wasted retries]")
result_c_status = result_c["status"]
print(f"  Status: {result_c_status}")
result_c_attempts_used = result_c["attempts_used"]
print(f"  Attempts used: {result_c_attempts_used} (should be 1, NOT 4)")
print(f"  Real elapsed time: {elapsed_c:.3f}s (should be near-instant, no backoff delays)")

if result_c["attempts_used"] == 1 and elapsed_c < elapsed_b:
    print(f"\nCONFIRMED: the permanent error failed on attempt 1 with near-zero delay,")
    print(f"while the exhausted-transient-retry case (B) consumed the FULL retry")
    print(f"budget and backoff time ({elapsed_b:.3f}s) -- exactly the theory's claim")
    print(f"that conflating these categories would waste real, measurable time.")

print("\nModule 2 complete. Run Module 3 next.")


REAL RETRY-WITH-BACKOFF, MEASURED TIMING

[Case A: fails twice, succeeds on 3rd attempt]
  Status: success
  Attempts used: 3
  Real elapsed time: 0.600s (includes REAL backoff delays)
  Result: {'found': True, 'reference_number': 'BJ2019FD7717', 'Customer_Name': 'Shobha Chopra', 'Status': 'Closed (Premature)'}

[Case B: fails 10 times -- exceeds retry budget of 4 attempts]
  Status: retries_exhausted
  Attempts used: 4
  Real elapsed time: 1.200s (full retry budget consumed)

[Case C: PERMANENT error -- should fail FAST, no wasted retries]
  Status: permanent_failure
  Attempts used: 1 (should be 1, NOT 4)
  Real elapsed time: 0.000s (should be near-instant, no backoff delays)

CONFIRMED: the permanent error failed on attempt 1 with near-zero delay,
while the exhausted-transient-retry case (B) consumed the FULL retry
budget and backoff time (1.200s) -- exactly the theory's claim
that conflating these categories would waste real, measurable time.

Module 2 complete. Run Module 3 next.


### Module 3: The Distinct Error Result Fed Back to the Model

Shows the three genuinely different result shapes (success, legitimate not-found, exhausted-retry error) and how each would be packaged as a tool_result using Topic 2's real message mechanics.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Distinct result shapes fed back to the model
# ------------------------------------------------------------------

import json

def build_tool_result_block(retry_outcome: dict, tool_use_id: str) -> dict:
    """Builds the ACTUAL tool_result block that would be sent back to
    the model, using Topic 2's real message mechanics -- but shaping
    the CONTENT differently depending on which of the three outcomes
    actually occurred, so the model can reason about them differently."""
    if retry_outcome["status"] == "success":
        content = retry_outcome["result"]  # either found=True or found=False -- BOTH legitimate
    elif retry_outcome["status"] == "permanent_failure":
        content = {"error": "invalid_request", "detail": retry_outcome["error"]}
    else:  # retries_exhausted
        content = {"error": "lookup_unavailable",
                   "detail": "Could not complete this check after multiple attempts. Please inform the customer of a temporary system issue."}

    return {"type": "tool_result", "tool_use_id": tool_use_id, "content": json.dumps(content)}


print("=" * 70)
print("THREE DISTINCT tool_result SHAPES SENT BACK TO THE MODEL")
print("=" * 70)

# reuse results from Module 2
scenarios = [
    ("Legitimate found=True (success)", {"status": "success", "result": {"found": True, "reference_number": "BJ2019FD7717", "Customer_Name": "Shobha Chopra"}}),
    ("Legitimate found=False (success, negative)", {"status": "success", "result": {"found": False, "reference_number": "BJ9999FD0000"}}),
    ("Retries exhausted (genuine system failure)", result_b),
    ("Permanent/malformed error (never retried)", result_c),
]

for label, outcome in scenarios:
    tool_result = build_tool_result_block(outcome, tool_use_id="toolu_test")
    print(f"\n[{label}]")
    tool_result_content = tool_result["content"]
    print(f"  tool_result sent to model: {tool_result_content}")

print("\nNotice all FOUR scenarios produce a DIFFERENT, explicit, honest")
print("content shape -- the model can distinguish a genuine record, a")
print("legitimate not-found, an unavailable-system state, and an invalid")
print("request from each other, and reason (and respond to the customer)")
print("appropriately for each, rather than these being silently collapsed")
print("into one ambiguous 'something went wrong' signal.")

print("\nModule 3 complete. All Chapter 10 Topic 5 modules done.")
print()
print("=" * 70)
print("CHAPTER 10 TOPIC 5 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  THREE distinct failure categories, each needing DIFFERENT handling:
  legitimate negative result (never retry), transient error (retry with
  backoff), permanent error (never retry, fail fast).

  Retry logic needs a MAXIMUM attempt count and INCREASING delay
  between attempts -- demonstrated with real measured timing showing
  the exhausted-retry case consumes real, bounded time.

  Permanent errors must fail FAST, with zero wasted retry attempts --
  demonstrated concretely: 1 attempt, near-zero elapsed time, vs the
  full retry budget consumed for a genuinely transient, unresolved
  failure.

  Every outcome (success/found=True, success/found=False, exhausted-
  retries error, permanent error) needs its OWN distinct, honest result
  shape fed back to the model -- never collapse them into one signal.
""")


THREE DISTINCT tool_result SHAPES SENT BACK TO THE MODEL

[Legitimate found=True (success)]
  tool_result sent to model: {"found": true, "reference_number": "BJ2019FD7717", "Customer_Name": "Shobha Chopra"}

[Legitimate found=False (success, negative)]
  tool_result sent to model: {"found": false, "reference_number": "BJ9999FD0000"}

[Retries exhausted (genuine system failure)]
  tool_result sent to model: {"error": "lookup_unavailable", "detail": "Could not complete this check after multiple attempts. Please inform the customer of a temporary system issue."}

[Permanent/malformed error (never retried)]
  tool_result sent to model: {"error": "invalid_request", "detail": "Malformed query for 'malformed-input' -- will fail every time."}

Notice all FOUR scenarios produce a DIFFERENT, explicit, honest
content shape -- the model can distinguish a genuine record, a
legitimate not-found, an unavailable-system state, and an invalid
request from each other, and reason (and respond to the custo